# Build new inference index

This notebook shows how to build an Index compatible with `opr.inference.index`.


In [ ]:
from pathlib import Path
from IPython.display import display

from torchvision import transforms as T
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn, Tensor
import open3d as o3d
import MinkowskiEngine as ME
from PIL import Image


In [ ]:
DIST_THRESH = 0.5
ANGLE_THRESH_DEG = 30

POINTCLOUD_QUANTIZATION_SIZE = 0.05


The example below is for SberRobotics Office dataset.

- `map1` - database, sampled with `DIST_THRESH` OR `ANGLE_THRESH_DEG`
- The rest of the maps will be the queries

In [ ]:
root_data_dir = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed"
)
assert root_data_dir.exists(), f"Path {root_data_dir} does not exist"

DB_MAP_DIR = root_data_dir / "map1" / "keyframe_map"
assert DB_MAP_DIR.exists(), f"Path {DB_MAP_DIR} does not exist"


In [ ]:
!ls -l $DB_MAP_DIR

In [ ]:
!head $DB_MAP_DIR/poses.csv

## load poses.csv

In [ ]:
def load_trajectory_df(traj_path, sep=","):
    traj_df = pd.read_table(
        traj_path, header=None, sep=sep,
        names=["timestamp", "x", "y", "z", "qx", "qy", "qz", "qw"],
        comment='#'
    )
    print(f"Loaded frames with {len(traj_df)} entries.")
    return traj_df

In [ ]:
db_map_df = load_trajectory_df(DB_MAP_DIR / "poses.csv")
display(db_map_df.head())
display(db_map_df.tail())

## Check images dir

In [ ]:
images_dir = DB_MAP_DIR / "zedx_front_left" / "rgb"
assert images_dir.exists(), f"Path {images_dir} does not exist"

num_files = len(list(images_dir.iterdir()))
print(f"Number of files: {num_files}")

images_timestamps = DB_MAP_DIR / "zedx_front_left" / "image_timestamps.txt"

images_timestamps = pd.read_csv(images_timestamps, header=None, sep=", ")
images_timestamps = images_timestamps[1].tolist()
images_timestamps = np.array([x / 1e9 for x in images_timestamps], dtype=np.float64)

print(images_timestamps[:5])

lidar_timestamps = db_map_df["timestamp"].to_numpy(dtype=np.float64)

diffs_0 = np.abs(lidar_timestamps[:-2] - images_timestamps)
diffs_1 = np.abs(lidar_timestamps[1:-1] - images_timestamps)
diffs_2 = np.abs(lidar_timestamps[2:] - images_timestamps)

print(diffs_0.mean(), diffs_0.max(), diffs_0.min(), np.median(diffs_0))
print(diffs_1.mean(), diffs_1.max(), diffs_1.min(), np.median(diffs_1))
print(diffs_2.mean(), diffs_2.max(), diffs_2.min(), np.median(diffs_2))

## cut poses by images len

In [ ]:
if num_files < len(db_map_df):
    db_map_df = db_map_df.iloc[:num_files]

display(db_map_df.tail())


## filter poses

In [ ]:
def filter_db_by_spatial_and_angular_thresholds(
    df: pd.DataFrame,
    dist_thresh: float,
    angle_thresh_deg: float,
    position_columns: list[str] = ["x", "y", "z"],
    quaternion_columns: list[str] = ["qx", "qy", "qz", "qw"],
    start_with_first: bool = True,
) -> pd.DataFrame:
    """Filter frames by spatial and angular thresholds.

    Samples frames sequentially from ``df``. A frame is selected if it is farther
    than ``dist_thresh`` from all currently selected frames. Otherwise (if it is
    spatially close), it is selected only if the angular distance to the closest
    selected frame exceeds ``angle_thresh_deg``.

    Args:
        df: DataFrame with at least the position and quaternion columns.
        dist_thresh: Minimum Euclidean distance (meters) to accept unconditionally.
        angle_thresh_deg: Angular distance threshold in degrees when spatially close.
        position_columns: Names of position columns in order [x, y, z].
        quaternion_columns: Names of quaternion columns in order [qx, qy, qz, qw].
        start_with_first: Whether to force-include the first frame to seed selection.

    Returns:
        A new DataFrame containing the sampled frames in their original order.
    """
    # Validate columns
    required_cols = position_columns + quaternion_columns
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if len(df) == 0:
        return df.copy()

    positions = df[position_columns].to_numpy(dtype=float, copy=True)
    quaternions = df[quaternion_columns].to_numpy(dtype=float, copy=True)

    selected_indices: list[int] = []

    def _normalize_quaternion(q: np.ndarray) -> np.ndarray:
        """Return unit quaternion; if zero-norm, return input."""
        norm = np.linalg.norm(q)
        return q / norm if norm > 0 else q

    def _quaternion_angle_diff_deg(q1: np.ndarray, q2: np.ndarray) -> float:
        """Compute minimal angular difference between two quaternions in degrees."""
        q1n = _normalize_quaternion(q1)
        q2n = _normalize_quaternion(q2)
        dot = float(np.clip(np.abs(np.dot(q1n, q2n)), -1.0, 1.0))
        angle_rad = 2.0 * np.arccos(dot)
        return float(np.degrees(angle_rad))

    if start_with_first:
        selected_indices.append(0)

    selected_positions = (
        positions[selected_indices]
        if selected_indices
        else np.empty((0, positions.shape[1]), dtype=float)
    )
    selected_quaternions = (
        quaternions[selected_indices]
        if selected_indices
        else np.empty((0, quaternions.shape[1]), dtype=float)
    )

    start_i = 1 if start_with_first else 0
    for i in range(start_i, len(df)):
        pos_i = positions[i]
        quat_i = quaternions[i]

        if selected_positions.shape[0] == 0:
            selected_indices.append(i)
            selected_positions = np.vstack([selected_positions, pos_i])
            selected_quaternions = np.vstack([selected_quaternions, quat_i])
            continue

        dists = np.linalg.norm(selected_positions - pos_i, axis=1)
        nearest_sel_idx = int(np.argmin(dists))
        min_dist = float(dists[nearest_sel_idx])

        if min_dist > dist_thresh:
            should_select = True
        else:
            ang_deg = _quaternion_angle_diff_deg(quat_i, selected_quaternions[nearest_sel_idx])
            should_select = ang_deg > angle_thresh_deg

        if should_select:
            selected_indices.append(i)
            selected_positions = np.vstack([selected_positions, pos_i])
            selected_quaternions = np.vstack([selected_quaternions, quat_i])

    filtered_df = df.iloc[selected_indices]
    print(
        f"Selected {len(filtered_df)} / {len(df)} frames "
        f"({len(filtered_df) / max(1, len(df)):.1%})."
    )
    return filtered_df


In [ ]:
filtered_df = filter_db_by_spatial_and_angular_thresholds(
    db_map_df,
    dist_thresh=DIST_THRESH,
    angle_thresh_deg=ANGLE_THRESH_DEG,
)
display(filtered_df.head())
display(filtered_df.tail())

In [ ]:
def plot_xy_positions(df):
    """Plot 2D x-y positions from a DataFrame."""
    plt.figure(figsize=(8, 6))
    plt.scatter(df["x"], df["y"], s=8, alpha=0.7)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("2D Positions (x, y)")
    plt.axis("equal")
    plt.grid(True)
    plt.show()

plot_xy_positions(filtered_df)


In [ ]:
selected_indices = filtered_df.index.tolist()
print(f"selected {len(selected_indices)} indices, first 5: {selected_indices[:5]}")


## get paths for images files

In [ ]:
selected_images = [images_dir / f"{i:06d}.jpg" for i in selected_indices]
print(f"selected {len(selected_images)} images, first 5: {selected_images[:5]}")
selected_images_paths = [str(image.relative_to(DB_MAP_DIR)) for image in selected_images]
print(f"selected_images_paths: {selected_images_paths}")

scans = sorted((DB_MAP_DIR / "scans").glob("*.pcd"))
print(f"Found {len(scans)} scans")
selected_scans = [scans[i] for i in selected_indices]
print(f"selected {len(selected_scans)} scans, first 5: {selected_scans[:5]}")
selected_scans_paths = [str(scan.relative_to(DB_MAP_DIR)) for scan in selected_scans]
print(f"selected_scans_paths: {selected_scans_paths}")

## calculate descriptors for index

In [ ]:
from opr.models.place_recognition import MinkLoc3D
from opr.models.place_recognition.base import LateFusionModel
from mmpr.models import MegaLoc

lidar_model = MinkLoc3D()
lidar_model.load_state_dict(torch.load("../minkloc3d_nclt.pth"), strict=True)

rgb_model = MegaLoc()

model = LateFusionModel(image_module=rgb_model, cloud_module=lidar_model)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device);

In [ ]:
image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])


def read_image(image_filepath: str | Path) -> Tensor:
    image = Image.open(image_filepath)
    image = image_transform_fn(image)
    return image


def read_scan(scan_filepath: str | Path) -> tuple[np.ndarray, np.ndarray]:
    scan = o3d.io.read_point_cloud(str(scan_filepath))
    if not scan.has_points():
        raise ValueError(f"Scan file {scan_filepath} is empty or invalid.")
    # Convert to numpy array for easier manipulation
    scan = np.asarray(scan.points)
    coordinates = scan[:, :3]  # Get the first three columns (x, y, z)
    if scan.shape[1] == 3:
        features = np.ones((coordinates.shape[0], 1))
    elif scan.shape[1] == 4:
        features = scan[:, 3:4]  # Get the fourth column (intensity)
    else:
        raise ValueError(f"Unexpected scan format with shape {scan.shape}. Expected 3 or 4 columns.")
    return coordinates, features


def to_batch(coords: np.ndarray, feats: np.ndarray, image: Tensor) -> dict[str, Tensor]:
    coords_t = torch.from_numpy(coords).float()
    feats_t = torch.from_numpy(feats).float()
    quantized_coords, quantized_feats = ME.utils.sparse_quantize(
        coordinates=coords_t,
        features=feats_t,
        quantization_size=POINTCLOUD_QUANTIZATION_SIZE,
    )
    return {
        "pointclouds_lidar_coords": ME.utils.batched_coordinates([quantized_coords]),
        "pointclouds_lidar_feats": torch.cat([quantized_feats]),
        "images_0": image.unsqueeze(0),
    }


In [ ]:
descriptors = []
for image_path, scan_path in zip(selected_images_paths, selected_scans_paths):
    image = read_image(DB_MAP_DIR / image_path)
    scan_coords, scan_feats = read_scan(DB_MAP_DIR / scan_path)
    batch = to_batch(scan_coords, scan_feats, image)
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        desc = model(batch)
    descriptors.append(desc["final_descriptor"].cpu().numpy())
descriptors = np.concatenate(descriptors, axis=0)
np.save(DB_MAP_DIR / "descriptors.npy", descriptors)

print(f"Descriptors saved to {DB_MAP_DIR / 'descriptors.npy'}")


In [ ]:
N, D = descriptors.shape
print(f"descriptors shape: {descriptors.shape}")

## build FaissFlatIndex files

In [ ]:
# Build minimal FAISS index files for the filtered database and load FaissFlatIndex
import json
from opr.inference.index import FaissFlatIndex

index_dir = DB_MAP_DIR

# Build meta.parquet with required columns; pointcloud_path omitted (treated as NaN)
poses = filtered_df[["x", "y", "z", "qx", "qy", "qz", "qw"]].to_numpy(dtype=float)
poses_list = [list(p) for p in poses]
meta = pd.DataFrame({
    "idx": filtered_df.index.to_numpy(dtype=np.int64),
    "pose": poses_list,
    "image_path": selected_images_paths,
})
meta.to_parquet(index_dir / "meta.parquet")

# Minimal schema.json
schema = {"version": "1", "dim": D, "metric": "l2", "created_at": "", "opr_version": ""}
(index_dir / "schema.json").write_text(json.dumps(schema))

# Load the index
index = FaissFlatIndex.load(index_dir)
print(f"Index created at {index_dir}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")


# Sample PlaceRecognitionPipeline inference

In [ ]:
QUERY_MAP_DIR = root_data_dir / "map2" / "keyframe_map"
assert QUERY_MAP_DIR.exists(), f"Path {QUERY_MAP_DIR} does not exist"

image = read_image(QUERY_MAP_DIR / "zedx_front_left" / "rgb" / "000036.jpg")
scan_coords, scan_feats = read_scan(QUERY_MAP_DIR / "scans" / "000036.pcd")
sample_query_batch = to_batch(scan_coords, scan_feats, image)
sample_query_batch = {k: v.to(device) for k, v in sample_query_batch.items()}


In [ ]:
# Load index and run PlaceRecognitionPipeline
from opr.inference.index import FaissFlatIndex
from opr.inference.pipelines import PlaceRecognitionPipeline, SequencePlaceRecognitionPipeline

index = FaissFlatIndex.load(DB_MAP_DIR)
# pipeline = PlaceRecognitionPipeline(index=index, model=model, device="cuda")
pipeline = SequencePlaceRecognitionPipeline(index=index, model=model, device="cuda", max_window=23)

result = pipeline.infer(input_frame=sample_query_batch, k=5)
print(f"descriptor shape: {result.descriptor.shape}")
print(f"indices: {result.indices.tolist()}")
print(f"distances: {[float(x) for x in result.distances]}")
print(f"db_idx: {result.db_idx.tolist()}")
print(f"db_pose (first): {result.db_pose[0]}")

In [ ]:
from time import time

times = []
for i in range(300):
    image = read_image(QUERY_MAP_DIR / "zedx_front_left" / "rgb" / f"{i:06d}.jpg")
    scan_coords, scan_feats = read_scan(QUERY_MAP_DIR / "scans" / f"{i:06d}.pcd")
    sample_query_batch = to_batch(scan_coords, scan_feats, image)
    sample_query_batch = {k: v.to(device) for k, v in sample_query_batch.items()}
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    start = time()
    result = pipeline.infer(input_frame=sample_query_batch, k=5)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    end = time()
    times.append(end - start)

print(f"mean time: {np.mean(times[30:])*1000:.2f} ms")